# rmul-scalar-tensor-mix — worked example 3: __truediv__ and __rtruediv__ preserve operand order

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rmul-scalar-tensor-mix`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Division is NOT commutative, so the reflected `__rtruediv__` must NOT delegate to `__truediv__`. For `10 / box`, Python calls `box.__rtruediv__(10)`, which must compute `10 / box.value` (left operand over self), the reverse of what `__truediv__` does.

## Worked solution

`Frac` wraps a value. `__truediv__(self, other)` computes `self.value / other_v` (self over other). `__rtruediv__(self, other)` computes `other / self.value` (other over self) — the operand order is flipped because the wrapper is now on the RIGHT. We show `Frac(4) / 2 == 2.0` but `8 / Frac(4) == 2.0` via the reflected path, and crucially `2 / Frac(4) == 0.5`, not `2.0`, proving order is preserved. We print both directions.

In [ ]:
class Frac:
    def __init__(self, value):
        self.value = float(value)
    def __truediv__(self, other):
        ov = other.value if isinstance(other, Frac) else other
        return Frac(self.value / ov)
    def __rtruediv__(self, other):
        return Frac(other / self.value)  # other on LEFT: other / self
    def __repr__(self):
        return f'Frac({self.value})'


print('Frac(4) / 2 =', Frac(4) / 2)
print('8 / Frac(4) =', 8 / Frac(4))
print('2 / Frac(4) =', 2 / Frac(4))